# LangGraph Module · Day 08 — LangSmith Tracing

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Set up a **LangSmith project** and API key
- [ ] Enable **full tracing** on the LangGraph agent (the fraud-triage graph from Day 06/07)
- [ ] Analyse a trace: **latency**, **token usage**, **node execution order**
- [ ] Practice **reading and debugging** traces

**Python skill that feeds this (Day 08):** `structlog` — structured logging. A LangSmith **trace** is the same instinct as a structured log (emit *data*, correlate by an id) — but as a *tree* of runs instead of flat lines.


## 0. Setup

LangSmith is a **hosted** tracing dashboard — sending traces there needs an API key. But the *trace itself* is just a **tree of run records** that LangChain builds locally on every graph run. So this whole notebook runs **with no key**: we capture that exact tree offline with a `RunCollectorCallbackHandler` and read it ourselves. The final section ships the same trace to real LangSmith, guarded by `HAS_LANGSMITH`.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
# LangSmith reads either LANGSMITH_API_KEY (new) or LANGCHAIN_API_KEY (legacy)
HAS_LANGSMITH = bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY"))
HAS_GOOGLE = bool(os.getenv("GOOGLE_API_KEY"))

print("langsmith tracing notebook")
print("LangSmith key :", "loaded ✅" if HAS_LANGSMITH else "MISSING ❌ (only the final section needs it)")
print("Google key    :", "loaded ✅" if HAS_GOOGLE else "MISSING ❌ (only the real-LLM section needs it)")

langsmith tracing notebook
LangSmith key : loaded ✅
Google key    : loaded ✅


## 1. The one big idea — a **trace** is a *tree of runs*

Day 07 ended with `astream_events()`: a live firehose of every event a graph emits. Great in the moment, but it scrolls past and it's gone. A **trace** is that same information **captured, timed, nested, and saved** so you can inspect it *after* the run.

Every unit of work becomes a **run** with: a name, a type (`chain` / `llm` / `tool`), a start & end time (→ **latency**), inputs, outputs, and — for LLM runs — **token usage**. Runs **nest**: the graph is the root run, each node is a child, each model call is a child of its node. That tree, shared by one **run id**, is a trace.

Structured logs (today's Python skill) correlate flat lines by a `trace_id`. LangSmith does the same but keeps the **parent→child shape**, so you see not just *what* happened but *what happened inside what*.

## 2. Turning LangSmith on — it's **environment variables**, not code

The headline feature: you instrument **nothing**. LangChain/LangGraph already emit runs; LangSmith is enabled by setting a few env vars *before* your graph runs. Your agent code is byte-for-byte unchanged.

```bash
# put these in your .env (do NOT hard-code the key in a notebook)
LANGSMITH_API_KEY=lsv2_...            # from smith.langchain.com -> Settings
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=levelup-day08       # groups related runs in the dashboard
```

That's the entire "set up a project + API key" task: create a project name, paste your key, flip `LANGSMITH_TRACING=true`. Let's see what's currently set:

In [3]:
for k in ["LANGSMITH_TRACING", "LANGSMITH_PROJECT", "LANGSMITH_API_KEY", "LANGSMITH_API_KEY"]:
    v = os.getenv(k)
    shown = ("set ✅" if k.endswith("API_KEY") else v) if v else "unset"
    print(f"{k:20} = {shown}")

LANGSMITH_TRACING    = true
LANGSMITH_PROJECT    = LevelUp
LANGSMITH_API_KEY    = set ✅
LANGSMITH_API_KEY    = set ✅


☝ If those are unset, no traces leave your machine — and that's fine, the rest of the notebook needs no key. The key point to internalise: **tracing is a deploy-time switch (env vars), not a code change.** Same graph runs traced in prod and untraced in a unit test, with zero edits.

## 3. `@traceable` — instrument a plain Python function

LangChain objects trace themselves automatically. For your **own** helper functions (a fraud rule, a DB lookup), decorate them with **`@traceable`** and they become runs in the same tree. With no LangSmith key it's a **no-op wrapper** — the function runs and returns normally, so this cell works offline.

In [4]:
from langsmith import traceable

@traceable(run_type="tool", name="fraud_rule_check")
def fraud_rule_check(amount: float, country: str) -> dict:
    """A toy rule engine — becomes a 'tool' run in the trace."""
    reasons = []
    if amount > 1000: reasons.append("high_value")
    if country != "GB": reasons.append("overseas")
    return {"risk": "HIGH" if reasons else "LOW", "reasons": reasons}

print(fraud_rule_check(5000, "US"))   # runs normally with or without a key
print(fraud_rule_check(20, "GB"))

{'risk': 'HIGH', 'reasons': ['high_value', 'overseas']}
{'risk': 'LOW', 'reasons': []}


☝ Decorated once, `fraud_rule_check` is now a first-class node in any trace that calls it — you'd see its inputs (`amount`, `country`) and output (`risk`, `reasons`) in the dashboard. Offline it's transparent: same return values, no error. This is how you get *your* business logic to show up alongside the LLM calls instead of being an invisible black box between them.

## 4. Build the agent — and capture its trace **offline**

Here's the Day 06/07 fraud-triage graph: `triage` (rule-based) → `respond` (an LLM writes the customer message). We run it with a **`RunCollectorCallbackHandler`** attached — that handler collects the **exact same `Run` objects** LangSmith would receive, but keeps them in memory so we can read the trace with **no key**.

In [5]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

class TriageState(TypedDict):
    messages: Annotated[list, add_messages]
    risk: str

def _reply_model(text: str):
    # keyless stand-in for a real chat model (same idea as Day 07)
    return GenericFakeChatModel(messages=iter([AIMessage(content=text)]))

def triage(state: TriageState) -> dict:
    alert = state["messages"][-1].content.lower()
    risk = "HIGH" if ("overseas" in alert or "high-value" in alert) else "LOW"
    return {"risk": risk}

def respond(state: TriageState) -> dict:
    draft = (f"We spotted a {state['risk']}-risk transaction and paused it for review. "
             "A confirmation code has been sent to your phone.")
    reply = _reply_model(draft).invoke(state["messages"])
    return {"messages": [reply]}

b = StateGraph(TriageState)
b.add_node("triage", triage)
b.add_node("respond", respond)
b.add_edge(START, "triage")
b.add_edge("triage", "respond")
b.add_edge("respond", END)
graph = b.compile()
print("graph compiled: triage -> respond")

graph compiled: triage -> respond


In [6]:
from langchain_core.tracers.run_collector import RunCollectorCallbackHandler

collector = RunCollectorCallbackHandler()   # grabs the run tree locally

alert = {"messages": [HumanMessage(content="Overseas high-value purchase flagged")], "risk": ""}
result = graph.invoke(alert, config={"callbacks": [collector]})   # <-- attach the tracer

print("final risk :", result["risk"])
print("reply      :", result["messages"][-1].content)
print("runs captured:", len(collector.traced_runs))

final risk : HIGH
reply      : We spotted a HIGH-risk transaction and paused it for review. A confirmation code has been sent to your phone.
runs captured: 1


☝ `config={'callbacks': [collector]}` is the seam. In production LangSmith installs *its* callback here automatically from the env vars — same mechanism, different sink. `collector.traced_runs[0]` is the **root run** of the trace; everything the graph did hangs off it as children. Next we read that tree.

## 5. Read the trace — node order, latency, token usage

This is the "analyse a trace" task. We walk the run tree and print, for each run: its **type**, **name**, **latency** (end − start), and any **token usage**. The nesting shows **execution order** and *what ran inside what* — exactly the LangSmith timeline, in text.

In [7]:
root = collector.traced_runs[0]

def latency_ms(run):
    if run.end_time and run.start_time:
        return (run.end_time - run.start_time).total_seconds() * 1000
    return 0.0

def tokens(run):
    # LLM runs carry usage under outputs/extra; fake model reports none
    usage = (run.extra or {}).get("usage_metadata") if run.extra else None
    if not usage and run.outputs:
        usage = run.outputs.get("llm_output") or {}
    return usage or {}

def walk(run, depth=0):
    pad = "   " * depth
    line = f"{pad}[{run.run_type:5}] {run.name:22} {latency_ms(run):7.1f} ms"
    tok = tokens(run)
    if tok:
        line += f"   tokens={tok}"
    print(line)
    for child in run.child_runs:
        walk(child, depth + 1)

print("TRACE:", root.name, "\n" + "-" * 60)
walk(root)

TRACE: LangGraph 
------------------------------------------------------------
[chain] LangGraph                  5.4 ms
   [chain] triage                     0.4 ms
   [chain] respond                    1.8 ms
      [llm  ] GenericFakeChatModel       1.0 ms


☝ Read it top-down: the root **`LangGraph`** chain contains **`triage`** then **`respond`** — that ordering *is* the node execution order — and inside `respond` sits the **`llm`** run (the model call). The latency of each row tells you where time goes; in a real run the `llm` row dominates. Token usage is blank here because the fake model doesn't count tokens — the **field is there**, and §7 fills it with real Gemini numbers. This tree is precisely what LangSmith renders as a flame-graph timeline.

## 6. Debugging with a trace — the wrong node is obvious

Traces earn their keep when something's *wrong*. Say triage misfires and marks a clearly-overseas alert as `LOW`. In flat logs you'd guess; in the trace you open the **`triage`** run, read its **inputs and outputs**, and see the bad verdict immediately — without re-running anything.

In [8]:
# pull the triage run out of the tree and inspect its inputs -> outputs
triage_run = next(r for r in root.child_runs if r.name == "triage")
respond_run = next(r for r in root.child_runs if r.name == "respond")

print("triage  IN :", {"last_msg": triage_run.inputs["messages"][-1].content
                        if isinstance(triage_run.inputs.get("messages"), list) else triage_run.inputs})
print("triage  OUT:", triage_run.outputs)      # <- the risk verdict this node produced
print("respond OUT:", {"reply": respond_run.outputs["messages"][-1].content})

triage  IN : {'last_msg': 'Overseas high-value purchase flagged'}
triage  OUT: {'risk': 'HIGH'}
respond OUT: {'reply': 'We spotted a HIGH-risk transaction and paused it for review. A confirmation code has been sent to your phone.'}


☝ Each run stores its own **inputs and outputs**, so you can pinpoint exactly which node produced a bad value and what it saw. If `triage OUT` said `LOW` for an overseas alert, the bug is in the rule — not the LLM. That "open the offending run, read in/out" loop is 90% of real agent debugging, and it's why teams turn tracing on **before** they have problems, not after.

## 7. Real LangSmith — ship the trace to the dashboard (needs a key)

Everything above was offline. To get the hosted timeline, set the env vars from §2 and run the graph. LangSmith installs its own tracer from the environment — **no change to the graph**. This cell only runs if a LangSmith key is present; if you also have a Google key it traces a real Gemini call so you see genuine token counts.

In [9]:
if HAS_LANGSMITH:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "levelup-day08")

    from langsmith import Client
    from langchain_core.tracers.context import tracing_v2_enabled

    run_graph = graph
    if HAS_GOOGLE:
        # swap the fake responder for real Gemini so token usage is populated
        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

        def respond_real(state: TriageState) -> dict:
            prompt = (f"A {state['risk']}-risk card transaction was flagged. "
                      "Write a calm 2-sentence message to the customer.")
            return {"messages": [llm.invoke([HumanMessage(content=prompt)])]}

        rb = StateGraph(TriageState)
        rb.add_node("triage", triage); rb.add_node("respond", respond_real)
        rb.add_edge(START, "triage"); rb.add_edge("triage", "respond"); rb.add_edge("respond", END)
        run_graph = rb.compile()

    # tracing_v2_enabled captures the project + gives us the run URL
    with tracing_v2_enabled(project_name=os.environ["LANGSMITH_PROJECT"]) as cb:
        out = run_graph.invoke(
            {"messages": [HumanMessage(content="Overseas high-value purchase flagged")], "risk": ""}
        )
    print("risk:", out["risk"])
    print("reply:", out["messages"][-1].content)
    try:
        print("view trace ->", cb.get_run_url())     # clickable LangSmith link
    except Exception as e:
        print("(open smith.langchain.com -> project", os.environ["LANGSMITH_PROJECT"], "to view)")
else:
    print("No LangSmith key -> skipping. Set LANGSMITH_TRACING/LANGSMITH_API_KEY in .env to try it.")
    print("Everything you needed to learn today ran keyless in §1-6.")

risk: HIGH
reply: For your security, we've flagged a recent transaction on your card due to unusual activity. Please review your recent activity or contact us to confirm its legitimacy.
view trace -> https://smith.langchain.com/o/d076ef3d-776d-46a3-b318-5bf2197566ef/projects/p/a914e897-c217-4fab-9e89-81c45144b774/r/019f3aca-4b73-7943-bd7d-a2682f8542c1?poll=true


☝ With a key, `cb.get_run_url()` prints a link straight to this run's flame-graph in LangSmith: every node, its latency bar, the model's prompt/response, and **real token counts + cost**. Notice the graph definition never changed between the keyless run (§4) and this one — **tracing is bolted on by the environment**, exactly as promised in §2.

## 8. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **trace** | a tree of **runs** sharing one run id | see *what happened inside what*, saved & searchable |
| **run** | one unit of work: name, type, latency, in/out, tokens | the node/model/tool you inspect when debugging |
| **env-var setup** | `LANGSMITH_TRACING` + `LANGSMITH_API_KEY` + `LANGSMITH_PROJECT` | turn tracing on with **zero code change** |
| **`@traceable`** | decorator making *your* functions into runs | your business logic shows up beside the LLM calls |
| **`RunCollectorCallbackHandler`** | captures the run tree locally | inspect a real trace **offline / in tests**, no key |
| **latency / order / tokens** | fields on each run | find the slow node, the wrong verdict, the token hog |
| **`tracing_v2_enabled` / `get_run_url`** | scoped tracing + link to the run | jump straight to the dashboard timeline |


